# A Multi-Witness Complex for Multi-Resource Accessibility

**Construction 2** from the discussion of O'Neil & Tymochko, *Evaluating Cooling-Center
Coverage Using Persistent Homology of a Filtered Witness Complex* (arXiv:2410.09067).

## The idea

The paper builds **one** filtered witness complex from a single resource: landmarks
$L=\{\ell_i\}$ are census-block centroids, witnesses $W=\{w_j\}$ are cooling centers, and
an edge $\{\ell_i,\ell_j\}$ appears at scale $\alpha$ when **some** witness lies within
$\alpha$ of **both** endpoints:

$$\{\ell_i,\ell_j\}\in\mathcal{W}(L,W)_\alpha \iff \exists\,w\in W:\ d(\ell_i,w)\le\alpha \ \text{and}\ d(\ell_j,w)\le\alpha .$$

Two blocks joined by an edge are read as having *equal coverage* by that resource.

To study **multi-resource** accessibility we keep the same landmark set $L$ but use **two**
witness families — resource $A$ (e.g. cooling centers, $W^A$) and resource $B$
(e.g. transit stops, $W^B$). The **multi-witness complex** requires an edge to be
witnessed **in both families at once** (the AND / co-coverage rule):

$$\{\ell_i,\ell_j\}\in\mathcal{W}(L,W^A,W^B)_\alpha \iff
\big(\exists\,w\in W^A:\ d(\ell_i,w),d(\ell_j,w)\le\alpha\big)\ \textbf{ and }\
\big(\exists\,w'\in W^B:\ d(\ell_i,w'),d(\ell_j,w')\le\alpha\big).$$

Note $w$ and $w'$ **need not be the same point** — each block must be able to reach *a*
cooling center *and* (separately) *a* transit stop. This is exactly the **intersection of
the two single-resource witness complexes**, $\mathcal{W}(L,W^A)_\alpha \cap
\mathcal{W}(L,W^B)_\alpha$, so it is a valid single-parameter filtration and drops straight
into GUDHI.

Everything below uses plain **Euclidean distance** and a small toy layout.

> Swapping AND $\to$ OR gives the **union** $\mathcal{W}(L,W^A)_\alpha \cup
> \mathcal{W}(L,W^B)_\alpha$ = "served by *either* resource". We compute both so the
> contrast is visible.


In [ ]:
import numpy as np
import gudhi
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon
from matplotlib.lines import Line2D
from itertools import combinations

print("gudhi", gudhi.__version__)

## 1. Toy layout

Eight census-block centroids sit on a ring (so a 1-D coverage *loop* can exist). We split
the two resources so each serves a **different half** of the ring — resource $A$ witnesses
sit at the midpoints of the "even" ring edges, resource $B$ at the "odd" ones. This is the
interesting regime: neither resource alone covers the whole perimeter cheaply, and the
AND-rule exposes where co-coverage is missing.

In [ ]:
n = 8
ang = np.linspace(0, 2*np.pi, n, endpoint=False)
R = 3.0
L = np.column_stack([R*np.cos(ang), R*np.sin(ang)])          # landmarks = block centroids

r_mid = R*np.cos(np.pi/n)                                     # radius of ring-edge midpoints
even_mid = ang[[0, 2, 4, 6]] + np.pi/n                        # midpoints of edges (0,1)(2,3)(4,5)(6,7)
odd_mid  = ang[[1, 3, 5, 7]] + np.pi/n                        # midpoints of edges (1,2)(3,4)(5,6)(7,0)

WA = np.column_stack([r_mid*np.cos(even_mid), r_mid*np.sin(even_mid)])   # resource A (cooling centers)
WB = np.column_stack([r_mid*np.cos(odd_mid),  r_mid*np.sin(odd_mid)])    # resource B (transit)

def euclid(a, b):
    return float(np.hypot(*(a - b)))

CA, CB, CAND, COR = "#2c7fb8", "#d95f0e", "#31a354", "#756bb1"

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(L[:, 0], L[:, 1], s=140, c="#f7f7f7", edgecolors="#4d4d4d", linewidths=1.4, zorder=5)
for i, (x, y) in enumerate(L):
    ax.annotate(fr"$\ell_{{{i}}}$", (x, y), ha="center", va="center", fontsize=9, zorder=6)
ax.scatter(WA[:, 0], WA[:, 1], marker="D", s=70, c=CA, edgecolors="white", label="resource A (cooling centers)")
ax.scatter(WB[:, 0], WB[:, 1], marker="s", s=70, c=CB, edgecolors="white", label="resource B (transit)")
ax.set_aspect("equal"); ax.axis("off")
ax.legend(loc="center", fontsize=8, frameon=False)
ax.set_title("Toy layout: 8 block centroids + two resource families", fontsize=10)
plt.show()

## 2. Edge birth scales

For a single witness set $W$, the scale at which edge $\{\ell_i,\ell_j\}$ first appears is

$$\mathrm{grade}(i,j;W)=\min_{w\in W}\ \max\big(d(\ell_i,w),\,d(\ell_j,w)\big),$$

i.e. the smallest $\alpha$ for which *some* witness is within $\alpha$ of both endpoints.

- **AND (multi-witness, construction 2):** the edge needs *both* families, so it is born at
  the **later** of the two grades — $\max(\mathrm{grade}_A,\mathrm{grade}_B)$.
- **OR (union):** *either* family suffices, so it is born at the **earlier** —
  $\min(\mathrm{grade}_A,\mathrm{grade}_B)$.

In [ ]:
def edge_grade(i, j, W):
    return min(max(euclid(L[i], w), euclid(L[j], w)) for w in W)

edges = list(combinations(range(n), 2))
grA   = {e: edge_grade(*e, WA) for e in edges}
grB   = {e: edge_grade(*e, WB) for e in edges}
grAND = {e: max(grA[e], grB[e]) for e in edges}   # multi-witness (intersection)
grOR  = {e: min(grA[e], grB[e]) for e in edges}   # union

print("edge : gradeA gradeB  AND    OR")
for e in [(0, 1), (1, 2), (2, 3), (0, 2), (0, 4)]:
    print(f"{e} : {grA[e]:.2f}  {grB[e]:.2f}   {grAND[e]:.2f}  {grOR[e]:.2f}")

## 3. Build the filtered flag complexes

The witness complex is a **flag (clique) complex**: once all edges of a simplex are present
it is filled in. GUDHI's `SimplexTree.expansion(k)` does exactly this, assigning each
higher simplex the max filtration value of its edges. We build four complexes — resource
$A$ alone, resource $B$ alone, their **union** (OR) and their **intersection** (AND =
construction 2) — and compute persistent homology for each.

In [ ]:
def build_flag_complex(grades, n_pts, max_dim=2):
    st = gudhi.SimplexTree()
    for v in range(n_pts):
        st.insert([v], filtration=0.0)          # landmarks all born at alpha = 0
    for (i, j), f in grades.items():
        st.insert([i, j], filtration=f)
    st.expansion(max_dim)                        # flag/clique expansion
    st.compute_persistence()
    return st

complexes = {
    "A only":            build_flag_complex(grA,   n),
    "B only":            build_flag_complex(grB,   n),
    "OR (union)":        build_flag_complex(grOR,  n),
    "AND (constr. 2)":   build_flag_complex(grAND, n),
}
diagrams = {k: st.persistence() for k, st in complexes.items()}

for k, dg in diagrams.items():
    h0 = sorted(d for dim, (b, d) in dg if dim == 0 and d != np.inf)
    h1 = [(round(b, 2), round(d, 2)) for dim, (b, d) in dg if dim == 1]
    print(f"{k:16s} H0 finite deaths = {[round(x,2) for x in h0]}   H1 = {h1}")

## 4. Filtration evolution

Each row is a construction; each column a scale $\alpha$. Discs are the $\alpha$-balls
around witnesses; an edge is drawn when its endpoints are co-witnessed. Rows 3 and 4 show
the **union (OR)** and **intersection (AND)** of the two resources side by side. Contrast
them: under **OR** the ring closes early (a block is served if it can reach *either*
resource), while under the multi-witness **AND** rule the blocks stay isolated far
longer — the coverage loop only appears once *both* resources reach across each ring
edge.

In [ ]:
def triangle_grades(grades):
    tg = {}
    for t in combinations(range(n), 3):
        es = [tuple(sorted(p)) for p in combinations(t, 2)]
        tg[t] = max(grades[e] for e in es)
    return tg

tgA, tgB, tgOR, tgAND = triangle_grades(grA), triangle_grades(grB), triangle_grades(grOR), triangle_grades(grAND)

def draw_complex(ax, grades, tgrades, alpha, edge_color):
    for t, f in tgrades.items():
        if f <= alpha:
            ax.add_patch(Polygon(L[list(t)], closed=True, color=edge_color, alpha=0.18, lw=0, zorder=1))
    for (i, j), f in grades.items():
        if f <= alpha:
            ax.plot([L[i, 0], L[j, 0]], [L[i, 1], L[j, 1]], color=edge_color, lw=2.2, zorder=2)
    ax.scatter(L[:, 0], L[:, 1], s=90, c="#f7f7f7", edgecolors="#4d4d4d", linewidths=1.2, zorder=5)
    ax.set_aspect("equal"); ax.set_xlim(-4.6, 4.6); ax.set_ylim(-4.6, 4.6); ax.axis("off")

def draw_two_family(ax, gr, tg, al, ec):
    for w in WA:
        ax.add_patch(Circle(w, al, color=CA, alpha=0.05, lw=0, zorder=0))
    for w in WB:
        ax.add_patch(Circle(w, al, color=CB, alpha=0.05, lw=0, zorder=0))
    draw_complex(ax, gr, tg, al, ec)
    ax.scatter(WA[:, 0], WA[:, 1], marker="D", s=36, c=CA, edgecolors="white", linewidths=0.6, zorder=4)
    ax.scatter(WB[:, 0], WB[:, 1], marker="s", s=36, c=CB, edgecolors="white", linewidths=0.6, zorder=4)

alphas = [1.0, 1.8, 2.3, 3.9]
rows = [("Resource A only",     grA,   tgA,   WA,   CA, CA),
        ("Resource B only",     grB,   tgB,   WB,   CB, CB),
        ("A OR B (union)",      grOR,  tgOR,  None, None, COR),
        ("A AND B (constr. 2)", grAND, tgAND, None, None, CAND)]

fig, axs = plt.subplots(len(rows), len(alphas), figsize=(3.0*len(alphas), 3.0*len(rows)))
for ri, (rname, gr, tg, W, wc, ec) in enumerate(rows):
    for ci, al in enumerate(alphas):
        ax = axs[ri, ci]
        if ri < 2:                                   # single-resource rows
            for w in W:
                ax.add_patch(Circle(w, al, color=wc, alpha=0.09, lw=0, zorder=0))
            draw_complex(ax, gr, tg, al, ec)
            ax.scatter(W[:, 0], W[:, 1], marker="D", s=42, c=wc, edgecolors="white", linewidths=0.6, zorder=4)
        else:                                        # OR and AND rows: both families shown
            draw_two_family(ax, gr, tg, al, ec)
        if ri == 0:
            ax.set_title(fr"$\alpha={al}$", fontsize=11)
        if ci == 0:
            ax.text(-0.05, 0.5, rname, transform=ax.transAxes, rotation=90,
                    va="center", ha="right", fontsize=9)
fig.suptitle("Filtration evolution: two single-resource witness complexes, their union (OR) and intersection (AND)", y=0.995)
fig.tight_layout()
fig.savefig("fig_filtration_evolution.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Persistence diagrams

$H_0$ points (circles) track connected components — how long each block stays isolated;
$H_1$ points (triangles) track coverage loops. Points near the diagonal are short-lived;
points high above it are persistent (robust) features. Compare **OR** and **AND**: under OR
the blocks merge early (low $H_0$ deaths); under the multi-witness AND rule they stay
separate far longer, and the coverage loop is born later and lives longer.

In [ ]:
order = ["A only", "B only", "OR (union)", "AND (constr. 2)"]
colmap = {"A only": CA, "B only": CB, "OR (union)": COR, "AND (constr. 2)": CAND}

finite = [d for dg in diagrams.values() for _, (b, d) in dg if d != np.inf]
births = [b for dg in diagrams.values() for _, (b, d) in dg]
vmax = max(finite + births); inf_y = vmax*1.12

fig, axs = plt.subplots(1, 4, figsize=(15, 4.0))
for ax, k in zip(axs, order):
    c = colmap[k]
    ax.plot([0, vmax*1.05], [0, vmax*1.05], color="#bdbdbd", ls="--", lw=1, zorder=0)
    ax.axhline(inf_y, color="#bdbdbd", ls=":", lw=0.8)
    for dim, (b, d) in diagrams[k]:
        y = inf_y if d == np.inf else d
        ax.scatter(b, y, marker=("o" if dim == 0 else "^"), s=55,
                   facecolors=("white" if dim == 0 else c), edgecolors=c, linewidths=1.4, zorder=3)
    ax.text(vmax*0.02, inf_y*1.005, r"$\infty$", fontsize=9, va="bottom", color="#666")
    ax.set_title(k, fontsize=10, color=c); ax.set_xlabel(r"birth $\alpha$")
    ax.set_xlim(-0.1, vmax*1.08); ax.set_ylim(-0.1, inf_y*1.05)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axs[0].set_ylabel(r"death $\alpha$")
axs[-1].legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec="#444", label=r"$H_0$ (components)"),
                        Line2D([], [], marker="^", ls="", mfc="#444", mec="#444", label=r"$H_1$ (loops)")],
               loc="lower right", fontsize=7)
fig.suptitle("Persistence diagrams", y=1.02)
fig.tight_layout()
fig.savefig("fig_persistence_diagrams.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Persistence barcodes — OR vs AND

Each horizontal bar is one homology class, drawn from birth to death along $\alpha$. Thin
bars are $H_0$ (a block joining the rest of the city); the thick bar is the $H_1$ coverage
loop. The **multi-witness AND** rule keeps every block's bar alive much longer — the
signature of *stricter* co-coverage: a block counts as "connected" only once it can reach
**both** resource types.

In [ ]:
def barcode(ax, dg, color, title, inf_x):
    bars = sorted(((dim, b, d) for dim, (b, d) in dg),
                  key=lambda t: (t[0], t[1], inf_x if t[2] == np.inf else t[2]))
    for y, (dim, b, d) in enumerate(bars):
        x1 = inf_x if d == np.inf else d
        ax.plot([b, x1], [y, y], color=color, lw=(3.5 if dim == 0 else 6),
                solid_capstyle="butt", alpha=(0.85 if dim == 0 else 1.0), zorder=(2 if dim == 0 else 3))
    ax.set_xlim(-0.1, inf_x*1.02); ax.set_ylim(-1, len(bars))
    ax.set_yticks([]); ax.set_xlabel(r"$\alpha$"); ax.set_title(title, fontsize=10, color=color)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)

infx = vmax*1.08
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
barcode(axs[0], diagrams["OR (union)"],      COR,  "OR (union): served by either resource", infx)
barcode(axs[1], diagrams["AND (constr. 2)"], CAND, "AND (construction 2): served by both resources", infx)
for ax in axs:
    ax.text(infx*0.99, ax.get_ylim()[1]-1.3, r"thick bar = $H_1$ loop", fontsize=7, ha="right", color="#333")
fig.suptitle(r"Persistence barcodes  —  $H_0$ bars (thin) = block connection; $H_1$ bar (thick) = coverage loop", y=1.01)
fig.tight_layout()
fig.savefig("fig_barcodes.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Reading the result

- **Union (OR)** — a block is "covered" if it can reach *either* resource. Blocks merge
  early; the coverage loop closes quickly. This is a lenient view of accessibility.
- **Intersection (AND) = the multi-witness complex** — a block is "covered" only if it can
  reach *both* resource types. Blocks stay isolated far longer and the coverage loop
  persists, surfacing exactly the places where *joint* access is missing even though each
  resource individually looks fine.

**How to adapt to real data.** Replace `L` with block/tract centroids and `WA`, `WB` with
two real resource layers (lat/lon). For geographic honesty swap `euclid` for a geodesic
(great-circle) distance, exactly as O'Neil & Tymochko do — the rest of the pipeline is
unchanged. Long $H_0$ bars (or, in a map, large $H_0$/$H_1$ death values) flag the blocks
most in need of the missing joint coverage. Add a third resource by taking a further
intersection with $\mathcal{W}(L,W^C)_\alpha$.

*If you want each resource to have its **own** scale (independent $\alpha,\beta$) rather
than one shared $\alpha$, that is the richer **bifiltration** / 2-parameter persistence
object — analyze it with `multipers`/RIVET rather than ordinary PH.*